# 15 — Attention Mechanism desde Cero
**Goal:** Entender el mecanismo de atención que está en el corazón de BERT, GPT y todos los transformers — implementado con NumPy puro.

## La intuición

Dado el token `banco` en la frase:
- *"fui al **banco** a cobrar"* → `banco` debería atender más a `cobrar`
- *"me senté en el **banco** del parque"* → `banco` debería atender más a `parque`

La atención le permite al modelo usar el contexto para desambiguar.

## Scaled Dot-Product Attention (Vaswani et al., 2017)

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)\!V$$

- $Q$ = queries: "¿qué estoy buscando?"
- $K$ = keys: "¿qué ofrezco para ser encontrado?"
- $V$ = values: "si me encuentran, esto es lo que entrego"
- $\sqrt{d_k}$: escala para evitar gradientes pequeños con dimensiones altas

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
})

np.random.seed(42)
print('Solo NumPy. Vamos.')

## 1. Softmax y scaled dot-product attention

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x = x - x.max(axis=axis, keepdims=True)   # stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (seq_len_q, d_k)
    K: (seq_len_k, d_k)
    V: (seq_len_k, d_v)
    Returns: (seq_len_q, d_v), attention_weights (seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]

    # Scores: cada query con cada key  → (seq_len_q, seq_len_k)
    scores = Q @ K.T / np.sqrt(d_k)

    # Máscara (ej: para no atender tokens de padding o tokens futuros)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)

    # Attention weights: softmax sobre la dimensión de las keys
    weights = softmax(scores, axis=-1)   # (seq_len_q, seq_len_k)

    # Output: suma ponderada de los values
    output  = weights @ V                # (seq_len_q, d_v)

    return output, weights

# Ejemplo mínimo: 4 tokens, d_model=8
seq_len, d_k, d_v = 4, 8, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)
print('Output shape:  ', output.shape)    # (4, 8)
print('Weights shape: ', weights.shape)   # (4, 4)
print('Weights suma a 1:', weights.sum(axis=-1).round(3))

## 2. Atención sobre una oración — ejemplo concreto

In [ ]:
# Simular embeddings para 'la app está lenta'
# En la práctica, estos vienen de una lookup table entrenada
tokens = ['la', 'app', 'está', 'lenta']
d_model = 16

rng = np.random.default_rng(1)
embeddings = rng.standard_normal((len(tokens), d_model))  # (4, 16)

# Proyecciones lineales Q, K, V (matrices entrenables en la práctica)
W_Q = rng.standard_normal((d_model, d_model))
W_K = rng.standard_normal((d_model, d_model))
W_V = rng.standard_normal((d_model, d_model))

Q = embeddings @ W_Q   # (4, 16)
K = embeddings @ W_K
V = embeddings @ W_V

output, weights = scaled_dot_product_attention(Q, K, V)

# Visualizar pesos de atención
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=11)
ax.set_xlabel('Keys (tokens atendidos)')
ax.set_ylabel('Queries (token actual)')
ax.set_title('Attention weights — "la app está lenta"')
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='white' if weights[i,j] > 0.5 else 'black')
plt.tight_layout()
plt.show()

## 3. Multi-Head Attention

In [ ]:
class MultiHeadAttention:
    """
    Multi-Head Attention (Vaswani et al. 2017).
    Permite que el modelo atienda a información desde distintas posiciones
    simultáneamente usando múltiples 'cabezas' de atención.
    """

    def __init__(self, d_model, n_heads, rng=None):
        assert d_model % n_heads == 0
        rng = rng or np.random.default_rng(42)

        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads   # dimensión por cabeza

        # Una proyección Q, K, V por cabeza
        scale = np.sqrt(2 / d_model)
        self.W_Q = [rng.standard_normal((d_model, self.d_k)) * scale for _ in range(n_heads)]
        self.W_K = [rng.standard_normal((d_model, self.d_k)) * scale for _ in range(n_heads)]
        self.W_V = [rng.standard_normal((d_model, self.d_k)) * scale for _ in range(n_heads)]

        # Proyección de salida
        self.W_O = rng.standard_normal((d_model, d_model)) * scale

    def forward(self, X, mask=None):
        """
        X: (seq_len, d_model)
        Returns: (seq_len, d_model), list of attention_weights per head
        """
        head_outputs = []
        all_weights  = []

        for h in range(self.n_heads):
            Q = X @ self.W_Q[h]   # (seq_len, d_k)
            K = X @ self.W_K[h]
            V = X @ self.W_V[h]
            out, w = scaled_dot_product_attention(Q, K, V, mask)
            head_outputs.append(out)
            all_weights.append(w)

        # Concatenar cabezas → (seq_len, d_model)
        concat = np.concatenate(head_outputs, axis=-1)

        # Proyección final
        output = concat @ self.W_O
        return output, all_weights

# Demo con 4 cabezas
mha = MultiHeadAttention(d_model=16, n_heads=4, rng=np.random.default_rng(42))
output_mha, weights_list = mha.forward(embeddings)

print(f'Input shape:  {embeddings.shape}')
print(f'Output shape: {output_mha.shape}')
print(f'N heads: {len(weights_list)} × weights shape: {weights_list[0].shape}')

In [ ]:
# Visualizar las 4 cabezas de atención
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for h, (w, ax) in enumerate(zip(weights_list, axes)):
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, fontsize=9)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(f'Head {h+1}', fontweight='bold')
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center',
                    fontsize=8, color='white' if w[i,j] > 0.5 else 'black')

plt.suptitle('Multi-Head Attention — cada cabeza aprende un patrón distinto', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Causal mask — GPT-style (solo atender al pasado)

In [ ]:
# En decoders (GPT): un token solo puede ver tokens anteriores
# Se implementa con una máscara triangular inferior

seq_len = 5
causal_mask = np.tril(np.ones((seq_len, seq_len), dtype=bool))  # triangular inferior

print('Causal mask (True = puede atender):')
print(causal_mask.astype(int))

# Aplicar a la atención
X5 = np.random.default_rng(0).standard_normal((seq_len, d_model))
Q5 = X5 @ W_Q; K5 = X5 @ W_K; V5 = X5 @ W_V
_, weights_causal = scaled_dot_product_attention(Q5[:, :8], K5[:, :8], V5[:, :8], mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

_, w_full = scaled_dot_product_attention(Q5[:, :8], K5[:, :8], V5[:, :8])

for ax, w, title in [(axes[0], w_full, 'Atención completa (BERT-style)'),
                      (axes[1], weights_causal, 'Atención causal (GPT-style)')]:
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=0.5)
    plt.colorbar(im, ax=ax)
    labels = [f't{i}' for i in range(seq_len)]
    ax.set_xticks(range(seq_len)); ax.set_xticklabels(labels)
    ax.set_yticks(range(seq_len)); ax.set_yticklabels(labels)
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 5. Positional encoding — inyectar información de orden

In [ ]:
def positional_encoding(seq_len, d_model):
    """
    Encoding sinusoidal (Vaswani et al. 2017).
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    PE = np.zeros((seq_len, d_model))
    positions = np.arange(seq_len)[:, None]              # (seq_len, 1)
    dims      = np.arange(0, d_model, 2)                 # [0, 2, 4, ...]
    angles    = positions / (10000 ** (dims / d_model))  # (seq_len, d_model/2)
    PE[:, 0::2] = np.sin(angles)
    PE[:, 1::2] = np.cos(angles)
    return PE

# Visualizar
PE = positional_encoding(seq_len=50, d_model=64)

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(PE.T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xlabel('Posición en la secuencia')
ax.set_ylabel('Dimensión del embedding')
ax.set_title('Positional Encoding sinusoidal (Vaswani et al. 2017)')
plt.tight_layout()
plt.show()

# Propiedad: posiciones cercanas tienen encodings similares
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(PE)
print('Similitud coseno entre posiciones:')
print(f'  pos 0 vs 1:  {sim[0,1]:.3f}')
print(f'  pos 0 vs 5:  {sim[0,5]:.3f}')
print(f'  pos 0 vs 25: {sim[0,25]:.3f}')
print(f'  pos 0 vs 49: {sim[0,49]:.3f}')

## 6. Transformer encoder — bloque completo

In [ ]:
def layer_norm(x, eps=1e-6):
    """Layer normalization — normaliza por la última dimensión."""
    mean = x.mean(axis=-1, keepdims=True)
    std  = x.std(axis=-1,  keepdims=True)
    return (x - mean) / (std + eps)

def feed_forward(x, W1, b1, W2, b2):
    """FFN: ReLU(xW1 + b1)W2 + b2"""
    return np.maximum(0, x @ W1 + b1) @ W2 + b2

class TransformerEncoderBlock:
    """Un bloque encoder del transformer (sin parámetros entrenados — solo arquitectura)."""

    def __init__(self, d_model=16, n_heads=4, d_ff=32, rng=None):
        rng = rng or np.random.default_rng(42)
        s   = np.sqrt(2 / d_model)

        self.mha = MultiHeadAttention(d_model, n_heads, rng)

        # FFN weights
        self.W1 = rng.standard_normal((d_model, d_ff)) * s
        self.b1 = np.zeros(d_ff)
        self.W2 = rng.standard_normal((d_ff, d_model)) * s
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        """
        x: (seq_len, d_model)
        Returns: (seq_len, d_model)
        """
        # Sub-layer 1: Multi-Head Attention + residual + LayerNorm
        attn_out, _ = self.mha.forward(x)
        x = layer_norm(x + attn_out)          # Add & Norm

        # Sub-layer 2: Feed-Forward + residual + LayerNorm
        ff_out = feed_forward(x, self.W1, self.b1, self.W2, self.b2)
        x = layer_norm(x + ff_out)            # Add & Norm

        return x

# Forward pass completo
encoder = TransformerEncoderBlock(d_model=16, n_heads=4, d_ff=32)
PE = positional_encoding(len(tokens), d_model=16)

x_in  = embeddings + PE           # embedding + positional encoding
x_out = encoder.forward(x_in)

print(f'Input shape:  {x_in.shape}')
print(f'Output shape: {x_out.shape}')
print('El shape se preserva — podemos apilar N bloques encoders.')

## 7. Arquitectura completa — roadmap

```
Input tokens
    ↓
Token Embedding  (lookup table: vocab_size × d_model)
    ↓
+ Positional Encoding
    ↓
┌─────────────────────────────────┐  ×N capas
│  Multi-Head Self-Attention      │
│      + Add & LayerNorm          │
│  Feed-Forward Network (2 capas) │
│      + Add & LayerNorm          │
└─────────────────────────────────┘
    ↓
Output representations (contextual embeddings)
    ↓
Task head: clasificación / generación / etc.
```

| Modelo | Tipo | Capas | d_model | Heads | Parámetros |
|---|---|---|---|---|---|
| BERT-base | Encoder | 12 | 768 | 12 | 110M |
| GPT-2 small | Decoder | 12 | 768 | 12 | 117M |
| GPT-4 (est.) | Decoder | ~96 | ~12288 | ~96 | ~1.7T |

## Resumen
| Componente | Fórmula |
|---|---|
| Scaled dot-product attention | $\text{softmax}(QK^T/\sqrt{d_k})V$ |
| Multi-head attention | Concatenar $h$ cabezas de atención paralelas |
| Causal mask | Máscara triangular inferior — bloquea tokens futuros |
| Positional encoding | $\sin/\cos$ sinusoidal — inyecta información de posición |
| Encoder block | MHA → Add&Norm → FFN → Add&Norm |
| Layer norm | $(x - \mu) / (\sigma + \epsilon)$ sobre la última dimensión |

**Siguiente:** `16_production.ipynb` — transformers sklearn personalizados, serialización y despliegue.